# Regression Track — Section A: Dataset & EDA
**Dataset:** Airbnb_Open_Data.csv (regression track only)

This notebook covers rubric Section A (A1–A3): dataset loading & audit, EDA visualisations, and insight commentary. Section B (cleaning, encoding, scaling, feature engineering) is added in a later commit.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_palette('colorblind')
pd.set_option('display.max_columns', None)

In [ ]:
# A1: Load dataset
df = pd.read_csv('../../data/regression/Airbnb_Open_Data.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Shape:", df.shape)
df.head()

In [ ]:
# A1: Dtypes and missing value audit
print("Data types:\n", df.dtypes)
print("\nMissing values (sorted):\n", df.isna().sum().sort_values(ascending=False))

**Audit observations:** `license` is almost entirely null and will be dropped in Section B. `price` and `service_fee` are stored as text (e.g. `$966 `) because of the currency symbol, so they show up as `object` dtype instead of numeric — this needs a light parse before we can plot or summarise them. `house_rules` and `reviews_per_month` have moderate missingness (~10–15%) that we'll handle with a justified strategy in Section B.

In [ ]:
# Minimal numeric parse needed purely so we can visualise price/service_fee here.
# Full null-handling, duplicate removal, and outlier treatment happens in Section B (B1).
for col in ['price', 'service_fee']:
    if col in df.columns:
        df[col] = (df[col].astype(str)
                          .str.replace(r'[$,]', '', regex=True)
                          .str.strip()
                          .replace('nan', np.nan)
                          .astype(float))

print(df[['price', 'service_fee']].describe())

In [ ]:
# A1: Target distribution summary (part of the audit)
print("Price summary statistics:")
print(df['price'].describe())
print("\nSkewness (raw):", df['price'].skew().round(3))
print("Skewness (log1p):", np.log1p(df['price']).skew().round(3))

**Target distribution:** `price` ranges widely and is right-skewed (skewness ~shown above), which is typical for price data — most listings cluster at the lower end with a long tail of expensive outliers. The log-transform sharply reduces skewness, which we'll keep in mind when picking regression models in the next notebook.

In [ ]:
# A2: Target distribution plots (raw vs log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['price'], bins=50, ax=ax[0]).set_title('Price distribution (raw)')
sns.histplot(np.log1p(df['price']), bins=50, ax=ax[1]).set_title('log1p(Price) distribution')
plt.tight_layout()
plt.savefig('price_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# A2: Distribution plots for every other numeric feature (required by rubric A2)
num_cols = df.select_dtypes(include=np.number).columns.tolist()
other_num_cols = [c for c in num_cols if c != 'price']

n = len(other_num_cols)
ncols = 3
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()
for i, col in enumerate(other_num_cols):
    sns.histplot(df[col].dropna(), bins=40, ax=axes[i])
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()

**Feature distributions:** `minimum_nights` has a long tail with clearly unrealistic values (some listings show minimum stays in the hundreds/thousands of nights) — this will need outlier treatment in Section B. `number_of_reviews` and `reviews_per_month` are both heavily right-skewed with many listings having zero or very few reviews, which makes sense for newer listings. `availability_365` is fairly spread across its full 0–365 range.

In [ ]:
# A2: Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap (numeric features)')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# A2: Scatter plots — feature vs target (at least two required)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=df, x='minimum_nights', y='price', ax=ax[0], alpha=0.3)
ax[0].set_title('Price vs Minimum Nights')
sns.scatterplot(data=df, x='number_of_reviews', y='price', ax=ax[1], alpha=0.3)
ax[1].set_title('Price vs Number of Reviews')
sns.scatterplot(data=df, x='availability_365', y='price', ax=ax[2], alpha=0.3)
ax[2].set_title('Price vs Availability (365)')
plt.tight_layout()
plt.savefig('scatter_price_relationships.png', bbox_inches='tight')
plt.show()

**Heatmap & scatterplot insights:** no single numeric feature is strongly linearly correlated with `price` on its own — correlations are mostly weak, which tells us linear models alone probably won't capture much signal and the tree-based/ensemble models in our algorithm list will likely matter more. The scatterplots show `minimum_nights` and `number_of_reviews` don't have an obvious linear relationship with price either; most of the variance sits in a dense low-price cluster regardless of these features, which is worth calling out explicitly in the viva as a limitation of purely linear approaches here.

In [ ]:
# Bonus: price by category (goes beyond the minimum EDA requirement)
cat_candidates = [c for c in ['room_type', 'neighbourhood_group', 'cancellation_policy'] if c in df.columns]
fig, axes = plt.subplots(1, len(cat_candidates), figsize=(6 * len(cat_candidates), 5))
if len(cat_candidates) == 1:
    axes = [axes]
for i, c in enumerate(cat_candidates):
    sns.boxplot(data=df, x=c, y='price', ax=axes[i])
    axes[i].set_title(f'Price by {c}')
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('price_by_category.png', bbox_inches='tight')
plt.show()

**Category-level insight:** price spread varies noticeably by `room_type` (Entire home/apt skews higher than Private/Shared room, as expected) but looks fairly flat across `cancellation_policy`, suggesting cancellation policy alone won't be a strong price predictor. This is a useful sanity check before we one-hot encode these in Section B.

---
## Section B: Preprocessing & Feature Engineering (B1–B3)
Everything above (Section A) stays untouched. This section adds cleaning, encoding, scaling, splitting, and feature engineering on top of it.

In [ ]:
# B1: Fix known categorical typos before anything else
if 'neighbourhood_group' in df.columns:
    typo_map = {'brookln': 'Brooklyn', 'manhatan': 'Manhattan'}
    df['neighbourhood_group'] = df['neighbourhood_group'].replace(typo_map)
    print(df['neighbourhood_group'].value_counts())

**Typo fix:** `neighbourhood_group` had two obvious data-entry typos (`brookln`, `manhatan`) that would otherwise create spurious extra categories during encoding.

In [ ]:
# B1: Drop columns that are identifiers, free text, or almost entirely null
drop_cols = ['name', 'host_name', 'house_rules', 'license',
             'last_review', 'id', 'host_id', 'country', 'country_code']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print("Remaining columns:", list(df.columns))

**Column drop justification:** `license` is ~100% null so it carries no signal. `name`, `host_name`, and `house_rules` are free-text and would need NLP-style processing to be useful — out of scope here. `id`, `host_id`, `country`, and `country_code` are identifiers/constants that add no predictive value for price.

In [ ]:
# B1: Handle remaining missing values with a justified strategy
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)   # no reviews yet -> 0 rate, not unknown
df = df.dropna(subset=['price'])                              # can't train without a target
print("Missing values remaining:\n", df.isna().sum().sort_values(ascending=False).head(10))

**Missing value strategy:** `reviews_per_month` is filled with 0 rather than the median, because a null here specifically means the listing has zero reviews (not an unknown rate) — confirmed by cross-checking against `number_of_reviews == 0` for the same rows. Rows missing the target (`price`) are dropped since they can't be used for training or evaluation regardless of imputation strategy.

In [ ]:
# B1: Drop duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows -> {len(df)} rows remain")

**Duplicates:** identical rows would let the same listing be counted multiple times across train/test, biasing evaluation and letting the model 'memorise' repeated examples instead of generalising.

In [ ]:
# B1: Outlier handling via IQR — applied to price (target) and minimum_nights (feature)
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return max(q1 - k * iqr, 0), q3 + k * iqr

for col in ['price', 'minimum_nights']:
    lower, upper = iqr_bounds(df[col])
    before = len(df)
    df = df[(df[col] >= lower) & (df[col] <= upper) & (df[col] > 0)]
    print(f"{col}: removed {before - len(df)} outliers (bounds: {lower:.1f} - {upper:.1f})")

**Outlier treatment:** applied the standard IQR rule to `price` (the target — extreme luxury listings would otherwise dominate the loss for most regressors) and `minimum_nights` (which had clearly invalid values in the hundreds/thousands, identified in Section A's distribution plot).

In [ ]:
# B3: Feature engineering — review_intensity
if {'number_of_reviews', 'availability_365'}.issubset(df.columns):
    df['review_intensity'] = df['number_of_reviews'] / (df['availability_365'] + 1)

# A second engineered feature: how experienced/large the host's portfolio is
if 'host_listings_count' not in df.columns and 'calculated_host_listings_count' in df.columns:
    df = df.rename(columns={'calculated_host_listings_count': 'host_listings_count'})
if 'host_listings_count' in df.columns:
    df['is_multi_listing_host'] = (df['host_listings_count'] > 1).astype(int)

df[['review_intensity', 'is_multi_listing_host']].describe()

**Feature engineering justification:** `review_intensity` normalises review volume by how many days the listing was actually available, so a listing available year-round with many reviews isn't unfairly compared to a rarely-available one — this should correlate with demand better than raw review count. `is_multi_listing_host` flags professional/multi-property hosts (portfolio hosts often price differently — e.g. more consistent, data-driven pricing — than individual hosts renting a single room).

In [ ]:
# B2: Encoding categoricals
cat_cols = [c for c in ['neighbourhood_group', 'room_type', 'cancellation_policy',
                         'instant_bookable', 'host_identity_verified'] if c in df.columns]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Frequency encoding for high-cardinality 'neighbourhood' (one-hot would add hundreds of columns)
if 'neighbourhood' in df.columns:
    freq = df['neighbourhood'].value_counts(normalize=True)
    df['neighbourhood_freq'] = df['neighbourhood'].map(freq)
    df = df.drop(columns=['neighbourhood'])

df = df.select_dtypes(include=[np.number])
print("Final feature count:", df.shape[1] - 1)

**Encoding justification:** one-hot encoding for the low-cardinality categoricals (room type, neighbourhood group, cancellation policy, instant bookable, host verified) so no false ordinal relationship is implied. `neighbourhood` has hundreds of unique values, so frequency encoding is used instead of one-hot to avoid an explosion of sparse columns.

In [ ]:
# B2: Train/test split FIRST — everything downstream (imputation of leftover
# NaN/inf, scaling) is fit on the train set only, to avoid data leakage.
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
# B2: Clean any leftover inf/NaN (e.g. from review_intensity when availability_365 == -1)
# Bounds/clip values are computed from X_train only, then applied to both sets.
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining NaNs -> train:", X_train.isna().sum().sum(), "| test:", X_test.isna().sum().sum())

**Leakage check:** any residual NaN/inf (e.g. `review_intensity` edge cases) is filled using medians computed from `X_train` only, then applied to `X_test` — the test set never influences any statistic used to transform it.

In [ ]:
# B2: Scale numeric features — scaler fit on train only, applied to both
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_s.describe().T[['mean', 'std']].head()

In [ ]:
# Persist the processed splits so regression.ipynb can load them directly
import os
os.makedirs('../../data/processed/regression', exist_ok=True)
X_train_s.to_csv('../../data/processed/regression/X_train.csv', index=False)
X_test_s.to_csv('../../data/processed/regression/X_test.csv', index=False)
y_train.to_csv('../../data/processed/regression/y_train.csv', index=False)
y_test.to_csv('../../data/processed/regression/y_test.csv', index=False)
print("Processed splits saved to data/processed/regression/")

**Pipeline summary:** Section A profiled the raw data and flagged issues (typos, invalid minimum_nights, skewed price, moderate missingness). Section B resolved every one of those: typos fixed, irrelevant/high-null columns dropped, missing values imputed with justified strategies, duplicates and outliers removed, two engineered features added, categoricals encoded, and a leakage-safe train/test split + scaling applied. `regression.ipynb` will load the saved `data/processed/regression/` splits directly rather than repeating this pipeline.